# TweetSense Sentiment Comparison

In this notebook we compare two methods:

1. **Traditional Machine Learning:** clean text, TF-IDF, then train ML classifiers.
2. **Local Hugging Face Model:** use a transformer zero-shot classifier.

Labels: `Positive`, `Negative`, `Neutral`, `Irrelevant`.

## 1. Import Libraries

In [ ]:
import re
import string
import time
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.tokenize import TweetTokenizer

sns.set_theme(style="whitegrid")

## 2. Basic Settings

In [ ]:
TRAIN_PATH = r"C:\Users\nwf15\OneDrive\Desktop\NLP_Project\twitter_training.csv"
VALIDATION_PATH = r"C:\Users\nwf15\OneDrive\Desktop\NLP_Project\twitter_validation.csv"

LABELS = ["Positive", "Negative", "Neutral", "Irrelevant"]
RANDOM_STATE = 42

MODEL_DIR = Path("models")
MODEL_DIR.mkdir(exist_ok=True)

print("Training file exists:", Path(TRAIN_PATH).exists())
print("Validation file exists:", Path(VALIDATION_PATH).exists())

## 3. Download NLTK Resources

We use NLTK for stopwords, stemming, and lemmatization.

In [ ]:
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

stop_words = set(stopwords.words("english"))
tokenizer = TweetTokenizer(preserve_case=False, strip_handles=True, reduce_len=True)
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

## 4. Load the Dataset

The dataset has four columns, so we give them clear names.

In [ ]:
columns = ["tweet_id", "entity", "sentiment", "tweet"]

train_df = pd.read_csv(TRAIN_PATH, header=None, names=columns)
valid_df = pd.read_csv(VALIDATION_PATH, header=None, names=columns)

print("Training shape:", train_df.shape)
print("Validation shape:", valid_df.shape)
train_df.head()

## 5. Clean Missing Values and Labels

In [ ]:
train_df = train_df.dropna(subset=["tweet", "sentiment"])
valid_df = valid_df.dropna(subset=["tweet", "sentiment"])

train_df = train_df[train_df["sentiment"].isin(LABELS)]
valid_df = valid_df[valid_df["sentiment"].isin(LABELS)]

print("Training labels")
display(train_df["sentiment"].value_counts())

print("Validation labels")
display(valid_df["sentiment"].value_counts())

## 6. Text Preprocessing Function

This follows our pipeline:

Raw tweet -> remove noise -> lowercase -> tokenize -> remove stopwords -> stemming or lemmatization.

In [ ]:
def clean_tweet(text, method="lemma"):
    text = str(text)

    # Remove URLs and mentions
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"@\w+", " ", text)

    # Remove only the hashtag symbol, but keep the word
    text = re.sub(r"#(\w+)", r"\1", text)

    # Remove emojis and non-English symbols
    text = text.encode("ascii", "ignore").decode("ascii")

    # Lowercase and remove punctuation
    text = text.lower()
    text = text.translate(str.maketrans("", "", string.punctuation))

    # Tokenize and remove stopwords
    tokens = tokenizer.tokenize(text)
    tokens = [word for word in tokens if word.isalpha() and word not in stop_words]

    # Normalize words
    if method == "stem":
        tokens = [stemmer.stem(word) for word in tokens]
    else:
        tokens = [lemmatizer.lemmatize(word) for word in tokens]

    return " ".join(tokens)


example = "I LOVE this update!!! 😍 https://example.com #Amazing @user"
print("Stemmed:", clean_tweet(example, method="stem"))
print("Lemmatized:", clean_tweet(example, method="lemma"))

## 7. Create Clean Text Columns

In [ ]:
train_df["tweet_stem"] = train_df["tweet"].apply(lambda x: clean_tweet(x, method="stem"))
valid_df["tweet_stem"] = valid_df["tweet"].apply(lambda x: clean_tweet(x, method="stem"))

train_df["tweet_lemma"] = train_df["tweet"].apply(lambda x: clean_tweet(x, method="lemma"))
valid_df["tweet_lemma"] = valid_df["tweet"].apply(lambda x: clean_tweet(x, method="lemma"))

train_df[["tweet", "tweet_stem", "tweet_lemma"]].head()

## 8. Train Traditional ML Models

We test three classifiers with both stemmed and lemmatized tweets.

In [ ]:
classifiers = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE),
    "Linear SVM": LinearSVC(class_weight="balanced", random_state=RANDOM_STATE),
    "Naive Bayes": MultinomialNB(),
}

text_versions = {
    "stem": ("tweet_stem", "Stemmed text"),
    "lemma": ("tweet_lemma", "Lemmatized text"),
}

traditional_results = []
saved_models = {}

for text_method, (text_column, text_name) in text_versions.items():
    X_train = train_df[text_column]
    X_valid = valid_df[text_column]
    y_train = train_df["sentiment"]
    y_valid = valid_df["sentiment"]

    for classifier_name, classifier in classifiers.items():
        model_name = f"{text_name} + {classifier_name}"
        print("Training:", model_name)

        model = Pipeline([
            ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_df=0.95)),
            ("classifier", classifier),
        ])

        start_time = time.time()
        model.fit(X_train, y_train)
        predictions = model.predict(X_valid)
        runtime = time.time() - start_time

        traditional_results.append({
            "method": "Traditional TF-IDF",
            "model": model_name,
            "text_method": text_method,
            "accuracy": accuracy_score(y_valid, predictions),
            "macro_f1": f1_score(y_valid, predictions, labels=LABELS, average="macro"),
            "weighted_f1": f1_score(y_valid, predictions, labels=LABELS, average="weighted"),
            "runtime_seconds": runtime,
        })

        saved_models[model_name] = model

traditional_results_df = pd.DataFrame(traditional_results).sort_values("macro_f1", ascending=False)
traditional_results_df

## 9. Evaluate and Save the Best Traditional Model

In [ ]:
best_row = traditional_results_df.iloc[0]
best_model_name = best_row["model"]
best_text_method = best_row["text_method"]
best_model = saved_models[best_model_name]

valid_clean_text = valid_df[f"tweet_{best_text_method}"]
traditional_predictions = best_model.predict(valid_clean_text)

print("Best traditional model:", best_model_name)
print(classification_report(valid_df["sentiment"], traditional_predictions, labels=LABELS))

model_package = {
    "model": best_model,
    "text_method": best_text_method,
    "labels": LABELS,
}

model_path = MODEL_DIR / "tweetsense_tfidf_model.joblib"
joblib.dump(model_package, model_path)
print("Saved model package to:", model_path)

In [ ]:
def show_confusion_matrix(y_true, y_pred, title):
    matrix = confusion_matrix(y_true, y_pred, labels=LABELS)

    plt.figure(figsize=(7, 5))
    sns.heatmap(matrix, annot=True, fmt="d", cmap="Blues", xticklabels=LABELS, yticklabels=LABELS)
    plt.title(title)
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.show()


show_confusion_matrix(valid_df["sentiment"], traditional_predictions, best_model_name)

## 10. Local Hugging Face Classification

We use a Hugging Face zero-shot model. This lets us classify tweets into our four labels without training a new neural network.

The first run may download the model. After it downloads, it runs locally from your computer cache.

In [ ]:
from transformers import pipeline

HF_MODEL_NAME = "facebook/bart-large-mnli"
HF_SAMPLE_PER_CLASS = 5

hf_sample = (
    valid_df
    .groupby("sentiment", group_keys=False)
    .apply(lambda group: group.sample(min(len(group), HF_SAMPLE_PER_CLASS), random_state=RANDOM_STATE))
    .sample(frac=1, random_state=RANDOM_STATE)
    .reset_index(drop=True)
)

print("Hugging Face sample size:", len(hf_sample))
hf_sample[["entity", "sentiment", "tweet"]].head()

In [ ]:
zero_shot_classifier = pipeline(
    "zero-shot-classification",
    model=HF_MODEL_NAME,
    device=-1,
)

def classify_with_hugging_face(tweet, entity):
    text = f"Tweet about {entity}: {tweet}"
    result = zero_shot_classifier(
        text,
        candidate_labels=LABELS,
        hypothesis_template="This tweet sentiment is {}.",
    )

    return result["labels"][0], result["scores"][0]


hf_predictions = []
hf_scores = []
start_time = time.time()

for _, row in hf_sample.iterrows():
    label, score = classify_with_hugging_face(row["tweet"], row["entity"])
    hf_predictions.append(label)
    hf_scores.append(score)

hf_runtime = time.time() - start_time

hf_sample["hf_prediction"] = hf_predictions
hf_sample["hf_score"] = hf_scores
hf_sample[["entity", "tweet", "sentiment", "hf_prediction", "hf_score"]].head()

## 11. Evaluate Hugging Face Model

In [ ]:
print(classification_report(hf_sample["sentiment"], hf_sample["hf_prediction"], labels=LABELS))
show_confusion_matrix(hf_sample["sentiment"], hf_sample["hf_prediction"], "Local Hugging Face Zero-Shot Model")

hf_metrics = {
    "method": "Local Hugging Face zero-shot",
    "model": HF_MODEL_NAME,
    "accuracy": accuracy_score(hf_sample["sentiment"], hf_sample["hf_prediction"]),
    "macro_f1": f1_score(hf_sample["sentiment"], hf_sample["hf_prediction"], labels=LABELS, average="macro"),
    "weighted_f1": f1_score(hf_sample["sentiment"], hf_sample["hf_prediction"], labels=LABELS, average="weighted"),
    "runtime_seconds": hf_runtime,
}

## 12. Final Comparison Table

In [ ]:
traditional_best_metrics = best_row.to_dict()
traditional_best_metrics["notes"] = "Trained and tested on the full validation set."

hf_metrics["notes"] = f"Tested on a small balanced sample of {len(hf_sample)} tweets because zero-shot models are slower."

comparison_df = pd.DataFrame([traditional_best_metrics, hf_metrics])
comparison_df[["method", "model", "accuracy", "macro_f1", "weighted_f1", "runtime_seconds", "notes"]]

## 13. Try the Saved Traditional Model

In [ ]:
loaded_package = joblib.load(model_path)
loaded_model = loaded_package["model"]
loaded_text_method = loaded_package["text_method"]

new_tweets = [
    "I love this game. The new update is amazing!",
    "This service is terrible and keeps crashing.",
    "The product page was updated today.",
    "I made lunch while everyone was talking about the match.",
]

clean_new_tweets = [clean_tweet(tweet, method=loaded_text_method) for tweet in new_tweets]
new_predictions = loaded_model.predict(clean_new_tweets)

pd.DataFrame({"tweet": new_tweets, "prediction": new_predictions})